# Eye-Tracking — Preprocessing & Classification Pipeline

**Output:** one `<participant_id>_cleaned_ET.csv` per participant, saved to `data/eye_tracking/processed/`

### Pipeline overview
1. **Discover** participant IDs from raw CSV filenames  
2. **Load & concatenate** all condition files (0, 1, 2, 3) for one participant  
3. **Check sampling rate** verify expected Hz and flag irregular intervals  
4. **Add relative time axis** (`time_ms` since first sample of each model)  
5. **Mark invalid samples** using left/right eye status fields  
6. **Interpolate** expand bad-sample windows ±2 frames, linear interp for bracketed gaps, re-normalise
7. **5-point median filter**
8. **Angular velocity** finite-difference angle between consecutive unit gaze vectors (deg/s)  
9. **MAD threshold** iterative Median Absolute Deviation; converges to participant-specific saccade threshold  
10. **Classify** each sample as `fixation` or `saccade`  
11. **Save** one CSV per participant  


## Data structure

```
eye-classification/
└── data/
    └── eye_tracking/
        ├── raw/
        │   ├── 001_ET_Data_Condition0_2026-05-06.csv
        │   ├── 001_ET_Data_Condition1_2026-05-06.csv
        │   ├── 001_ET_Data_Condition2_2026-05-06.csv
        │   ├── 001_ET_Data_Condition3_2026-05-06.csv
        │   ├── 002_ET_Data_Condition0_2026-05-06.csv
        │   └── ...
        └── processed/
            ├── 001_cleaned_ET.csv
            ├── 002_cleaned_ET.csv
            └── ...
```

# 1. Imports & Configuration

All imports, file paths, and algorithm parameters are collected here.  

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from matplotlib.colors import LinearSegmentedColormap

sns.set_style("whitegrid")

#  Paths 
DATA_DIR   = Path("../data/eye_tracking/raw")
OUTPUT_DIR = Path("../data/eye_tracking/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#  File layout
# Pattern: 001_ET_Data_Condition0_2026-05-06.csv  (conditions 0–3 per participant)
CONDITIONS = [0, 1, 2, 3]



FORWARD_COLS       = ["gaze_forward_x",      "gaze_forward_y",      "gaze_forward_z"]
CLEAN_FORWARD_COLS = ["clean_gaze_forward_x", "clean_gaze_forward_y", "clean_gaze_forward_z"]

#  Algorithm parameters 
EXPECTED_HZ = 200    # expected sampling rate (Hz)
VEL_MAX = 1000

#  Progress-bar format 
B_FORMAT = (
    "📄 {n_fmt} of {total_fmt} {desc} processed: {bar}\n"
    "    {percentage:3.0f}%  ⏱️ {elapsed}  ⏳ {remaining}  ⚙️ {rate_fmt}{postfix}"
)


In [2]:
print("Working directory:", Path.cwd())
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())
print()
print("All CSV files found:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(" ", f.name)

Working directory: /Users/azad/Desktop/HiWi/lego-vr-analysis/eye-classification/notebooks
DATA_DIR: ../data/eye_tracking/raw
DATA_DIR exists: True

All CSV files found:
  001_ET_Data_Condition0_2026-05-06.csv
  001_ET_Data_Condition1_2026-05-06.csv
  001_ET_Data_Condition2_2026-05-06.csv
  001_ET_Data_Condition3_2026-05-06.csv
  002_ET_Data_Condition0_2026-05-06.csv
  002_ET_Data_Condition1_2026-05-06.csv
  002_ET_Data_Condition2_2026-05-06.csv
  002_ET_Data_Condition3_2026-05-06.csv
  003_ET_Data_Condition0_2026-05-06.csv
  003_ET_Data_Condition1_2026-05-06.csv
  003_ET_Data_Condition2_2026-05-06.csv
  003_ET_Data_Condition3_2026-05-06.csv
  004_ET_Data_Condition0_2026-05-07.csv
  004_ET_Data_Condition1_2026-05-07.csv
  004_ET_Data_Condition2_2026-05-07.csv
  004_ET_Data_Condition3_2026-05-07.csv
  005_ET_Data_Condition0_2026-05-07.csv
  005_ET_Data_Condition1_2026-05-07.csv
  005_ET_Data_Condition2_2026-05-07.csv
  005_ET_Data_Condition3_2026-05-07.csv


# 2. Helper Functions

## 2.1 File Discovery

Scans `DATA_DIR` for CSVs matching `<id>_ET_Data_Condition<N>_*.csv`.  
Returns sorted unique participant IDs (e.g. `['001', '002', ...]`).  
`get_participant_files` returns one file path per condition for a given participant.

In [3]:
def get_participant_ids(data_dir=DATA_DIR):
    csv_files = sorted(data_dir.glob("*_ET_Data_Condition*_*.csv"))
    return sorted({file_path.name.split("_")[0] for file_path in csv_files})


def get_participant_files(participant_id, data_dir=DATA_DIR, conditions=CONDITIONS):
    files = []
    for condition in conditions:
        matches = sorted(data_dir.glob(f"{participant_id}_ET_Data_Condition{condition}_*.csv"))
        if matches:
            files.append(matches[0])
    return files


## 2.2 Data Loading

All condition files for one participant are concatenated into a single DataFrame.

In [4]:
def load_participant_minimal(file_paths):
    dfs = []
    for file_path in file_paths:
        df_part = pd.read_csv(file_path, low_memory=False)
        dfs.append(df_part)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


## 2.3 Sampling Rate Check

Before any preprocessing we verify the sampling rate.  

We report:
- **median interval** and **median Hz** : the actual average rate  
- **irregular samples** : frames where the interval deviates by more than 20% from expected  

This is diagnostic only: the data is not modified here.

In [5]:
df_raw = pd.read_csv(get_participant_files("001")[0], nrows=5)
print(df_raw["gaze_capture_time"].dtype)
print(df_raw["gaze_capture_time"].head())

int64
0    1000000972356065800
1    1000000972361063200
2    1000000972366064600
3    1000000972371070100
4    1000000972376071400
Name: gaze_capture_time, dtype: int64


In [6]:
df_raw = load_participant_minimal(get_participant_files("001"))
t = df_raw["gaze_capture_time"].to_numpy(dtype=np.int64)
dt_ms = np.diff(t) * 1e-6

print(f"Min interval  : {dt_ms.min():.2f} ms")
print(f"Max interval  : {dt_ms.max():.2f} ms")
print(f"Median        : {np.median(dt_ms):.2f} ms")
print(f"Jumps > 100ms : {(dt_ms > 100).sum()}")

Min interval  : 3.86 ms
Max interval  : 268668.53 ms
Median        : 5.00 ms
Jumps > 100ms : 529


In [7]:
def check_sampling_rate(df, participant_id, time_col="gaze_capture_time",
                        expected_hz=EXPECTED_HZ, min_hz=199):
    """
    Check sampling rate for one participant (overall and per trial).
    Inter-sample intervals are computed within each trial only.

    Parameters
    ----------
    expected_hz : int — expected sampling rate in Hz (used for display)
    min_hz      : int — trials with median Hz below this are flagged for exclusion

    Returns
    -------
    dict with overall stats + per-trial breakdown and list of excluded trials
    """

    # ── Overall (within-trial intervals only) ─────────────────────────────────
    all_intervals = np.concatenate([
        np.diff(sub[time_col].sort_values().to_numpy(dtype=np.int64)) * 1e-6
        for _, sub in df.groupby(["condition_number", "trial_number"], sort=True)
        if len(sub) >= 2
    ])

    overall_median_ms = float(np.median(all_intervals))
    overall_hz        = 1000.0 / overall_median_ms if overall_median_ms > 0 else np.nan
    status = "✅" if overall_hz >= min_hz else "⚠️"
    print(f"  {status}  Participant {participant_id}  —  overall: {overall_hz:.1f} Hz  ({overall_median_ms:.2f} ms/sample)")

    # ── Per trial ─────────────────────────────────────────────────────────────
    trial_rows      = []
    excluded_trials = []

    for (cond, trial), sub in df.groupby(["condition_number", "trial_number"], sort=True):
        t_trial = sub[time_col].sort_values().to_numpy(dtype=np.int64)
        if len(t_trial) < 2:
            trial_hz = np.nan
        else:
            dt_trial  = np.diff(t_trial) * 1e-6
            median_ms = float(np.median(dt_trial))
            trial_hz  = 1000.0 / median_ms if median_ms > 0 else np.nan

        excluded = np.isnan(trial_hz) or trial_hz < min_hz
        if excluded:
            excluded_trials.append((int(cond), int(trial)))
            print(f"      ❌  C{int(cond)}_T{int(trial):02d}  "
                  + (f"{trial_hz:.1f} Hz" if not np.isnan(trial_hz) else "NaN Hz"))

        trial_rows.append({
            "condition_number": int(cond),
            "trial_number":     int(trial),
            "median_hz":        round(trial_hz, 2) if not np.isnan(trial_hz) else np.nan,
            "n_samples":        len(sub),
            "excluded":         excluded,
        })

    if not excluded_trials:
        print(f"      all trials OK")
    print()

    return {
        "participant_id":  participant_id,
        "overall_hz":      round(overall_hz, 2),
        "overall_ms":      round(overall_median_ms, 3),
        "n_samples":       len(df),
        "trials":          trial_rows,
        "excluded_trials": excluded_trials,
    }

## 2.4 Relative Time Axis

Adds `time_ms`: milliseconds elapsed since the first sample of each Lego model.  
Sorting by `(condition_number, trial_number, gaze_capture_time)` first ensures the  
per-model `t0` anchor is always the true first sample.

In [8]:
def add_time_per_model(df, time_col="gaze_capture_time"):
    df = df.copy()
    df = df.sort_values(["condition_number", "trial_number", time_col])
    t0 = df.groupby("model_name")[time_col].transform("first")
    df["time_ms"] = (df[time_col] - t0) / 1_000_000.0
    return df


## 2.5 Status Marking & Interpolation

**`good_status`** : only frames where both eyes are `tracked` or `compensated` are valid.  
All other status combinations (including unilateral tracking loss) are flagged as bad.

**`find_runs` / `expand_runs`** : the Varjo tracker's state often transitions a frame or two  
*after* the actual onset/offset of an invalid period. Padding each bad-sample run by ±2 frames  
catches these edge artifacts before interpolation.

**`interpolate_segment`** : gaps that are fully bracketed by valid samples on both sides are  
filled with linear interpolation, then re-normalised to unit length.  
Unbracketed gaps (e.g. at segment start/end) are left as NaN.

**`interpolate_all_segments`** : applies the above to every `(condition, trial)` group  
and stamps each row with a `segment_label` (`C<cond>_T<trial>`).

In [9]:
def good_status(left_status, right_status):
    l = left_status.astype(str).str.lower().str.strip()
    r = right_status.astype(str).str.lower().str.strip()

    allowed_pairs = {
        ("tracked", "tracked"),
        ("tracked", "compensated"),
        ("compensated", "tracked"),
        ("compensated", "compensated"),
    }

    good = pd.Series(
        [(ls, rs) in allowed_pairs for ls, rs in zip(l, r)],
        index=left_status.index,
    )

    return good.to_numpy(dtype=bool)


def find_runs(mask):
    mask = np.asarray(mask, dtype=bool)
    idx = np.flatnonzero(mask)

    if len(idx) == 0:
        return []

    runs = []
    start = prev = idx[0]

    for k in idx[1:]:
        if k == prev + 1:
            prev = k
        else:
            runs.append((start, prev))
            start = prev = k

    runs.append((start, prev))
    return runs


def expand_runs(runs, n, pad=2):
    if len(runs) == 0:
        return []

    expanded = []
    for a, b in runs:
        aa = max(0, a - pad)
        bb = min(n - 1, b + pad)
        expanded.append((aa, bb))

    expanded.sort()
    merged = [expanded[0]]

    for a, b in expanded[1:]:
        prev_a, prev_b = merged[-1]
        if a <= prev_b + 1:
            merged[-1] = (prev_a, max(prev_b, b))
        else:
            merged.append((a, b))

    return merged


def normalize_xyz(xyz):
    xyz = np.asarray(xyz, dtype=float)
    nrm = np.linalg.norm(xyz, axis=1)
    out = np.full_like(xyz, np.nan, dtype=float)
    good = np.isfinite(nrm) & (nrm > 1e-8) & np.all(np.isfinite(xyz), axis=1)
    out[good] = xyz[good] / nrm[good, None]
    return out


def interpolate_segment(sub, forward_cols=FORWARD_COLS, pad=2):
    sub = sub.sort_values("time_ms").copy()
    t = sub["time_ms"].to_numpy(dtype=float)

    xyz = np.column_stack([
        pd.to_numeric(sub[c], errors="coerce").to_numpy(dtype=float)
        for c in forward_cols
    ])

    good = good_status(sub["left_status"], sub["right_status"])
    bad = ~good
    bad_runs = find_runs(bad)

    n = len(sub)
    is_interpolated = np.zeros(n, dtype=bool)
    clean = xyz.copy()

    buffered_runs = expand_runs(bad_runs, n=n, pad=pad)

    for a, b in buffered_runs:
        clean[a:b+1, :] = np.nan

    for a, b in buffered_runs:
        left = a - 1
        right = b + 1

        bracketed = (
            left >= 0 and
            right < n and
            np.all(np.isfinite(xyz[left])) and
            np.all(np.isfinite(xyz[right]))
        )

        if bracketed:
            for dim in range(3):
                clean[a:b+1, dim] = np.interp(
                    t[a:b+1],
                    [t[left], t[right]],
                    [xyz[left, dim], xyz[right, dim]],
                )

            clean[a:b+1, :] = normalize_xyz(clean[a:b+1, :])
            is_interpolated[a:b+1] = True

    sub["good_sample"] = good
    sub["bad_sample"] = bad
    sub["is_interpolated"] = is_interpolated

    for j, c in enumerate(forward_cols):
        sub[f"clean_{c}"] = clean[:, j]

    return sub


def interpolate_all_segments(df, pad=2):
    out = []

    for (cond, trial), sub in df.groupby(["condition_number", "trial_number"], sort=True):
        cleaned = interpolate_segment(sub, forward_cols=FORWARD_COLS, pad=pad)
        cleaned["segment_label"] = f"C{int(cond)}_T{int(trial)}"
        out.append(cleaned)

    return (
        pd.concat(out, axis=0)
        .sort_values(["condition_number", "trial_number", "time_ms"])
        .reset_index(drop=True)
    )


## 2.6 Median Smoothing (5-point)

A 5-point rolling median filter is applied to the clean gaze vectors before computing angular velocity.

- **Why median, not mean?** A median filter preserves sharp saccade onsets 
- **Why 5 points?** At 200 Hz, 5 frames = 25 ms. This is short enough to leave saccade timing intact (typical saccade duration > 30 ms) while still suppressing single-frame noise spikes.
- **Why per segment?** The filter runs independently within each `(condition, trial)` segment so that the boundary between two trials cannot bleed into either one.
- After filtering, gaze vectors are re-normalised to unit length since the median of unit vectors is not guaranteed to stay on the unit sphere.

In [10]:
def median_filter_gaze(df, cols=CLEAN_FORWARD_COLS, window=5):
    df = df.copy()

    for col in cols:
        df[col] = (
            df.groupby(["condition_number", "trial_number"])[col]
            .transform(lambda x: x.rolling(window=window, center=True, min_periods=1).median())
        )

    # re-normalise — median of unit vectors is not guaranteed to stay on the unit sphere
    xyz = df[cols].to_numpy(dtype=float)
    df[cols] = normalize_xyz(xyz)

    return df

## 2.7 Angular Velocity

Angular velocity (deg/s) is the angle between consecutive unit gaze vectors divided by inter-sample time.  
`arctan2(‖u × v‖, u · v)` is used instead of `arccos(u · v)`  (numerically stable for all angles),  
including very small ones where `arccos` loses precision near ±1.  
The first sample of each `(condition, trial)` segment is NaN (no previous frame to diff against).

In [11]:
def add_angular_velocity(df, time_col="gaze_capture_time"):
    df = df.copy()
    df = df.sort_values(["condition_number", "trial_number", time_col])
    df["angular_velocity"] = np.nan

    for (_, _), sub in df.groupby(["condition_number", "trial_number"], sort=False):
        idx = sub.index

        g = sub[CLEAN_FORWARD_COLS].to_numpy(dtype=float)
        norms = np.linalg.norm(g, axis=1, keepdims=True)
        g = g / np.clip(norms, 1e-12, None)

        g_prev = g[:-1]
        g_curr = g[1:]

        cross_norm = np.linalg.norm(np.cross(g_prev, g_curr), axis=1)
        dot_prod   = np.sum(g_prev * g_curr, axis=1)
        angles_rad = np.arctan2(cross_norm, dot_prod)

        t     = sub[time_col].to_numpy(dtype=np.int64)
        dt    = np.diff(t) * 1e-9
        vel   = np.full(len(sub), np.nan)
        valid = dt > 0
        vel[1:][valid] = np.degrees(angles_rad[valid] / dt[valid])

        # velocities above VEL_MAX are tracker glitches — set to NaN
        vel[vel > VEL_MAX] = np.nan

        df.loc[idx, "angular_velocity"] = vel

    return df

## 2.8 MAD Saccade Threshold

The Median Absolute Deviation algorithm finds a velocity threshold  
separating fixation from saccades (without assuming a fixed velocity distribution).

Starting from `th_0 = 200 deg/s`, it iteratively:
1. Keeps only velocities **below** the current threshold  
2. Computes `median + 3 × 1.486 × MAD` as the new threshold  
3. Stops when consecutive estimates differ by less than 1 deg/s  

The threshold is computed per participant to respect individual differences  
in eye movement amplitude and velocity.

In [12]:
def at_mad_threshold(angular_vel, th_0=200.0, min_samples=10, tol=1.0, max_iter=50):
    v = pd.to_numeric(pd.Series(angular_vel), errors="coerce").to_numpy(dtype=float)
    v = v[np.isfinite(v)]

    if len(v) < min_samples:
        return np.nan, []

    threshs = []
    current_th = float(th_0)

    for _ in range(max_iter):
        threshs.append(current_th)

        subset = v[v < current_th]
        subset = subset[np.isfinite(subset)]

        if len(subset) < min_samples:
            return np.nan, threshs

        median = np.median(subset)
        mad = np.median(np.abs(subset - median))
        next_th = median + 3.0 * 1.486 * mad

        if not np.isfinite(next_th):
            return np.nan, threshs

        if abs(current_th - next_th) <= tol:
            threshs.append(float(next_th))
            return float(next_th), threshs

        current_th = float(next_th)

    return float(current_th), threshs


## 2.9 Event Classification

Each sample is labelled `fixation` or `saccade` using the participant-specific MAD threshold:

| Label | Condition |
|-------|-----------|
| `saccade` | `angular_velocity > mad_threshold` |
| `fixation` | everything else (including NaN velocity) |


In [13]:
def classify_events(df, mad_threshold):
    """
    Label each sample as 'fixation' or 'saccade'.
    Parameters
    ----------
    mad_threshold : float — participant-specific saccade threshold (deg/s)
    """
    df = df.copy()
    df["mad_threshold"] = mad_threshold

    labels = np.full(len(df), "fixation", dtype=object)
    if np.isfinite(mad_threshold):
        labels[df["angular_velocity"].to_numpy() > mad_threshold] = "saccade"

    df["event_type"] = labels
    return df

# 3. Per-Participant Pipeline

`compute_participant_data` runs the full pipeline for one participant and returns  
the processed DataFrame plus a sampling-rate summary dict.  
Wrapping everything in one function keeps the main loop clean and makes it easy  
to re-run a single participant in isolation for debugging.

In [14]:
def compute_participant_data(participant_id):
    file_paths = get_participant_files(participant_id)
    df_part = load_participant_minimal(file_paths)

    if df_part.empty:
        return pd.DataFrame(), {
            "participant_id": participant_id,
            "n_files": 0,
            "source_files": [],
            "n_rows": 0,
            "n_valid_velocity_samples": 0,
            "mad_threshold": np.nan,
            "threshold_trace": [],
            "n_trace_steps": 0,
        }

    # Step 1 — sampling rate check (diagnostic only, does not modify data)
    sr_stats = check_sampling_rate(df_part, participant_id)

    # Step 2 — relative time axis
    df_part = add_time_per_model(df_part)

    # Step 3 — status marking + interpolation
    df_part = interpolate_all_segments(df_part, pad=2)

    # Step 4 — angular velocity
    df_part = add_angular_velocity(df_part, time_col="gaze_capture_time")

    # Step 5 — MAD threshold
    vel = pd.to_numeric(df_part["angular_velocity"], errors="coerce")
    vel = vel[np.isfinite(vel)]
    mad_threshold, threshold_trace = at_mad_threshold(vel, th_0=200.0)

    # Step 6 — classify
    df_part = classify_events(df_part, mad_threshold)
    df_part["participant_id"] = participant_id

    summary = {
        "participant_id":          participant_id,
        "n_files":                 len(file_paths),
        "source_files":            [path.name for path in file_paths],
        "n_rows":                  int(len(df_part)),
        "n_valid_velocity_samples": int(len(vel)),
        "mad_threshold":           mad_threshold,
        "threshold_trace":         threshold_trace,
        "n_trace_steps":           len(threshold_trace),
    }

    return df_part, summary


# 4. Run Pipeline — All Participants

Discovers all participant IDs, runs the full pipeline for each, and saves  
one `<participant_id>_cleaned_ET.csv` to `OUTPUT_DIR`.  
Results are processed one participant at a time to keep memory usage low.

In [15]:
participant_ids = get_participant_ids()
print(f"Found {len(participant_ids)} participant(s): {participant_ids}")


Found 5 participant(s): ['001', '002', '003', '004', '005']


In [16]:
threshold_summary_rows = []
failed                 = []

pbar = tqdm(
    participant_ids,
    desc="participants",
    bar_format=B_FORMAT,
    dynamic_ncols=True,
)

for participant_id in pbar:
    pbar.set_postfix_str(f"current → {participant_id}")
    try:
        df_part, summary = compute_participant_data(participant_id)

        if df_part.empty:
            failed.append(participant_id)
            continue

        out_path = OUTPUT_DIR / f"{participant_id}_cleaned_ET.csv"
        df_part.to_csv(out_path, index=False)
        threshold_summary_rows.append(summary)

    except Exception as exc:
        print(f"  ❌  {participant_id} failed: {exc}")
        failed.append(participant_id)

threshold_summary = (
    pd.DataFrame(threshold_summary_rows)
    .sort_values("participant_id")
    .reset_index(drop=True)
)

print(f"\n✅  Saved {len(threshold_summary_rows)} CSV(s) → {OUTPUT_DIR}")
if failed:
    print(f"❌  Failed / skipped: {failed}")


📄 0 of 5 participants processed:           
📄 0 of 5 participants processed:           
      0%  ⏱️ 00:00  ⏳ ?  ⚙️ ?it/s, current → 001

  ✅  Participant 001  —  overall: 200.0 Hz  (5.00 ms/sample)
      all trials OK



📄 1 of 5 participants processed: ██        
📄 1 of 5 participants processed: ██        urrent → 001
     20%  ⏱️ 00:21  ⏳ 01:24  ⚙️ 21.15s/it, current → 002

  ✅  Participant 002  —  overall: 200.0 Hz  (5.00 ms/sample)
      all trials OK



📄 2 of 5 participants processed: ████      
📄 2 of 5 participants processed: ████      urrent → 002
     40%  ⏱️ 00:40  ⏳ 01:00  ⚙️ 20.25s/it, current → 003

...

  ✅  Participant 004  —  overall: 200.0 Hz  (5.00 ms/sample)
      all trials OK



📄 4 of 5 participants processed: ████████  
📄 4 of 5 participants processed: ████████  urrent → 004
     80%  ⏱️ 01:20  ⏳ 00:20  ⚙️ 20.23s/it, current → 005

  ✅  Participant 005  —  overall: 200.0 Hz  (5.00 ms/sample)
      all trials OK



📄 5 of 5 participants processed: ██████████
📄 5 of 5 participants processed: ██████████urrent → 005
    100%  ⏱️ 01:42  ⏳ 00:00  ⚙️ 20.50s/it, current → 005


✅  Saved 5 CSV(s) → ../data/eye_tracking/processed
